In [1]:
df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df.show(5)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 3, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-01-01|Smartphone|Electronics|  600|      10|   6000|
|2022-01-01|    Laptop|Electronics| 1200|       5|   6000|
|2022-01-02|   T-Shirt|   Clothing|   20|      50|   1000|
|2022-01-03|Headphones|Electronics|  100|      20|   2000|
|2022-01-04|   T-Shirt|   Clothing|   20|      25|    500|
+----------+----------+-----------+-----+--------+-------+
only showing top 5 rows



In [2]:
# Remove null values
df_clean = df.dropna()

# Remove duplicate records
df_clean = df_clean.dropDuplicates()

# Preview cleaned data
df_clean.show(5)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 4, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-04-20|   T-Shirt|   Clothing|   20|      20|    400|
|2022-09-19|  Backpack|       Bags|   50|      15|    750|
|2022-01-01|    Laptop|Electronics| 1200|       5|   6000|
|2022-01-28|   Speaker|Electronics|   80|       8|    640|
|2022-06-27|Smartphone|Electronics|  600|       9|   5400|
+----------+----------+-----------+-----+--------+-------+
only showing top 5 rows



In [3]:
from pyspark.sql.functions import col

df_clean.printSchema()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 5, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- revenue: integer (nullable = true)



In [4]:
from pyspark.sql.functions import col, upper

df_clean = df_clean \
    .withColumn("product", upper(col("product"))) \
    .withColumn("category", upper(col("category")))

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 6, Finished, Available, Finished, False)

In [5]:
df_clean.write.mode("overwrite").saveAsTable("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 7, Finished, Available, Finished, False)

In [6]:
df_clean = df_clean.withColumn("total_value", col("price") * col("quantity"))

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.functions import sum, col

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.show()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 9, Finished, Available, Finished, False)

+-----------+-------------+--------------+
|   category|total_revenue|total_quantity|
+-----------+-------------+--------------+
|      SHOES|        20640|           258|
|ACCESSORIES|       101400|           902|
|   CLOHTING|         1200|            30|
|       BAGS|        19500|           390|
|    SHOESES|          960|            12|
|ELECTRONICS|       509480|          1439|
|   CLOTHING|        93150|          2221|
|       BGAS|          900|            18|
+-----------+-------------+--------------+



In [8]:
gold_df = gold_df.orderBy(col("total_revenue").desc())

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 10, Finished, Available, Finished, False)

In [9]:
gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import when, col

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "bgas", "BAGS")
    .when(col("category") == "clohting", "CLOTHING")
    .when(col("category") == "shoeses", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 12, Finished, Available, Finished, False)

In [11]:
df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 13, Finished, Available, Finished, False)

In [12]:
from pyspark.sql.functions import sum

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 14, Finished, Available, Finished, False)

In [13]:
df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df_clean = df.dropna().dropDuplicates()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import when, col, upper

df_clean = df_clean.withColumn("category", upper(col("category")))

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "BGAS", "BAGS")
    .when(col("category") == "CLOHTING", "CLOTHING")
    .when(col("category") == "SHOESES", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 16, Finished, Available, Finished, False)

In [15]:
df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 17, Finished, Available, Finished, False)

In [16]:
from pyspark.sql.functions import sum

gold_df = df_clean.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 18, Finished, Available, Finished, False)

In [17]:
from pyspark.sql.functions import col, to_date

df = spark.read.csv('Files/Bronze/sales_data.csv', header=True, inferSchema=True)

df = df.withColumn("date", to_date(col("date"), "M/d/yyyy"))

historical_df = df.filter(col("date") <= "2022-03-31")

historical_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 19, Finished, Available, Finished, False)

In [18]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-03-31") &
    (col("date") <= "2022-06-30")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 20, Finished, Available, Finished, False)

In [19]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-06-30") &
    (col("date") <= "2022-09-30")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 21, Finished, Available, Finished, False)

In [20]:
from pyspark.sql.functions import col

new_data_df = df.filter(
    (col("date") > "2022-09-30") &
    (col("date") <= "2022-12-31")
)


from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "bronze_sales")

(target.alias("old")
 .merge(
     new_data_df.alias("new"),
     "old.date = new.date AND old.product = new.product AND old.category = new.category"
 )
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 22, Finished, Available, Finished, False)

In [21]:
df_clean = spark.table("bronze_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 23, Finished, Available, Finished, False)

In [22]:
df_clean = df_clean.dropna()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 24, Finished, Available, Finished, False)

In [23]:
df_clean = df_clean.dropDuplicates()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 25, Finished, Available, Finished, False)

In [24]:
from pyspark.sql.functions import col, upper

df_clean = df_clean.withColumn("category", upper(col("category")))

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 26, Finished, Available, Finished, False)

In [25]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "category",
    when(col("category") == "BGAS", "BAGS")
    .when(col("category") == "CLOHTING", "CLOTHING")
    .when(col("category") == "SHOESES", "SHOES")
    .otherwise(col("category"))
)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 27, Finished, Available, Finished, False)

In [26]:
df_clean.select("category").distinct().show()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 28, Finished, Available, Finished, False)

+-----------+
|   category|
+-----------+
|      SHOES|
|ACCESSORIES|
|       BAGS|
|ELECTRONICS|
|   CLOTHING|
+-----------+



In [27]:
df_clean = df_clean.withColumn("revenue", col("price") * col("quantity"))

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 29, Finished, Available, Finished, False)

In [28]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .saveAsTable("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 30, Finished, Available, Finished, False)

In [29]:
spark.table("silver_sales").show()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 31, Finished, Available, Finished, False)

+----------+----------+-----------+-----+--------+-------+
|      date|   product|   category|price|quantity|revenue|
+----------+----------+-----------+-----+--------+-------+
|2022-01-01|    Laptop|ELECTRONICS| 1200|       5|   6000|
|2022-01-28|   Speaker|ELECTRONICS|   80|       8|    640|
|2022-03-12|Smartphone|ELECTRONICS|  600|      12|   7200|
|2022-02-07|   Speaker|ELECTRONICS|   80|      18|   1440|
|2022-01-19|   Speaker|ELECTRONICS|   80|      20|   1600|
|2022-03-18|   Speaker|ELECTRONICS|   80|      20|   1600|
|2022-01-09|   Speaker|ELECTRONICS|   80|      15|   1200|
|2022-01-03|Headphones|ELECTRONICS|  100|      20|   2000|
|2022-03-27|Smartphone|ELECTRONICS|  600|       9|   5400|
|2022-01-17|Smartphone|ELECTRONICS|  600|       6|   3600|
|2022-01-01|Smartphone|ELECTRONICS|  600|      10|   6000|
|2022-02-19|   Speaker|ELECTRONICS|   80|      12|    960|
|2022-02-22|Smartphone|ELECTRONICS|  600|       8|   4800|
|2022-01-22|Smartphone|ELECTRONICS|  600|       7|   420

In [30]:
spark.sql("SHOW PARTITIONS silver_sales").show()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 32, Finished, Available, Finished, False)

+--------------------+
|           partition|
+--------------------+
|category=ACCESSORIES|
|       category=BAGS|
|   category=CLOTHING|
|category=ELECTRONICS|
|      category=SHOES|
+--------------------+



In [31]:
df_gold = spark.table("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 33, Finished, Available, Finished, False)

In [32]:
from pyspark.sql.functions import sum

gold_df = df_gold.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 34, Finished, Available, Finished, False)

In [33]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 35, Finished, Available, Finished, False)

In [34]:
spark.table("gold_sales").show()

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 36, Finished, Available, Finished, False)

+-----------+-------------+--------------+
|   category|total_revenue|total_quantity|
+-----------+-------------+--------------+
|ELECTRONICS|       509480|          1439|
|   CLOTHING|        94350|          2251|
|ACCESSORIES|       101400|           902|
|       BAGS|        20400|           408|
|      SHOES|        21600|           270|
+-----------+-------------+--------------+



In [35]:
df_test = spark.table("silver_sales")

df_test = df_test.withColumn("revenue", col("revenue") + 10)

df_test.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("category") \
    .saveAsTable("silver_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 37, Finished, Available, Finished, False)

In [36]:
gold_df = df_test.groupBy("category").agg(
    sum("revenue").alias("total_revenue"),
    sum("quantity").alias("total_quantity")
)

gold_df.write.mode("overwrite").saveAsTable("gold_sales")

StatementMeta(, 6dda3979-bc8d-473a-865b-49b8ba87a09c, 39, Finished, Available, Finished, False)